In [ ]:
import os
import json
import pandas as pd
import traceback
import logging
import mcqgenerator.logger

In [ ]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

In [ ]:

load_dotenv()




In [ ]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.5,
    api_key=os.getenv("OPENAI_API_KEY")
)

Project Setup – Imports Explanation

-ChatOpenAI
This is the language model used to generate responses (e.g., MCQs).

-ChatPromptTemplate
Helps create structured prompts with placeholders (like {text} or {topic}).

-PyPDF2
Used to read and extract text from PDF files.

-langchain_community (optional)
Provides additional tools like callbacks (e.g., get_openai_callback) for tracking token usage and cost.

In [ ]:
!pip install langchain-community

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.callbacks import get_openai_callback
import PyPDF2 



quiz_template = """
Text:
{text}

You are an expert MCQ maker.
Given the above text, create {number} multiple choice questions
for {subject} students in a {tone} tone.

Make sure the questions:
- are not repeated
- match the text
- follow this JSON format

RESPONSE_JSON:
{response_json}
"""

quiz_prompt = ChatPromptTemplate.from_template(quiz_template)
quiz_chain = quiz_prompt | llm

review_template = """
You are an expert English grammarian and writer.

Given the following multiple choice quiz for {subject} students:

Quiz_MCQs:
{quiz}

Evaluate the complexity of the quiz in at most 50 words.
If needed, improve the wording and adjust the tone to fit the students' level.
"""

review_prompt = ChatPromptTemplate.from_template(review_template)
review_chain = review_prompt | llm

In [ ]:
# Step 1: get data
with open("data.txt", "r") as file:
    TEXT = file.read()



In [ ]:
response_json = """
{
  "1": {
    "mcq": "question here",
    "options": {
      "a": "option a",
      "b": "option b",
      "c": "option c",
      "d": "option d"
    },
    "correct": "a"
  }
}
"""

with get_openai_callback() as cb:
    try:
        quiz_result = quiz_chain.invoke({
            "text": "Your extracted PDF text goes here.",
            "number": 5,
            "subject": "biology",
            "tone": "simple",
            "response_json": response_json
        })
        logging.info("Quiz generated successfully")
    except Exception:
        logging.exception("Quiz generation failed")
        raise

    try:
        review_result = review_chain.invoke({
            "subject": "biology",
            "quiz": quiz_result.content
        })
        logging.info("Review generated successfully")
    except Exception:
        logging.exception("Review generation failed")
        raise

    try:
       quiz_dict = json.loads(quiz_result.content)
    except json.JSONDecodeError:
         logging.exception("Failed to parse quiz JSON")
         quiz_dict = None

    print("Generated Quiz:\n")
    print(quiz_dict)

    print("\nReview:\n")
    print(review_result.content)

    print("\nToken Usage:\n")
    print(f"Total Tokens: {cb.total_tokens}")
    print(f"Prompt Tokens: {cb.prompt_tokens}")
    print(f"Completion Tokens: {cb.completion_tokens}")
    print(f"Total Cost (USD): ${cb.total_cost}")


In [ ]:
import pandas as pd

quiz_table_data = []

if quiz_dict:
    for key, value in quiz_dict.items():
        mcq = value["mcq"]

        options = " | ".join(
            [
                f"{option}: {option_value}"
                for option, option_value in value["options"].items()
            ]
        )

        correct = value["correct"]

        quiz_table_data.append({
            "MCQ": mcq,
            "Choices": options,
            "Correct": correct
        })

# Convert to DataFrame
quiz_df = pd.DataFrame(quiz_table_data)

# Display nicely in notebook
quiz_df

In [ ]:
quiz.tocsv("quiz.csv", index=False)

In [ ]:
from datetime import datetime
datetime.now().strftime("%Y-%m-%d_%H-%M-%S.log")